# 09 Reset Data

## Purpose

Delete generated data files under the configured `DATA_DIR` so that the full notebook pipeline (`01` to `08`) can be re-run from a clean state. All directories are preserved; only files are removed.

The notebook defaults to a safe preview. A real deletion requires `DRY_RUN=false`. External data paths additionally require `ALLOW_EXTERNAL_DATA_RESET=true`.

## Inputs

All files currently present under `data/bronze/`, `data/silver/`, `data/gold/`, `data/samples/`, and `data/checkpoints/`.

## Outputs

Empty directory tree under `data/`. No Parquet, CSV, JSON, HTML, or JSONL files remain.
Re-running notebooks 01–08 in order will repopulate all directories.

## Configuration

The default is `DRY_RUN=true`: files are listed but not deleted.

Set `DRY_RUN=false` only when the preview is correct. If `DATA_DIR` points outside the repository, also set `ALLOW_EXTERNAL_DATA_RESET=true`. The notebook rejects filesystem roots and project-root deletion targets.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
configured_data_dir = Path(os.getenv("DATA_DIR", "data"))
DATA_DIR = (configured_data_dir if configured_data_dir.is_absolute() else PROJECT_ROOT / configured_data_dir).resolve()

DRY_RUN = os.getenv("DRY_RUN", "true").lower() == "true"
ALLOW_EXTERNAL_DATA_RESET = os.getenv("ALLOW_EXTERNAL_DATA_RESET", "false").lower() == "true"

assert DATA_DIR != PROJECT_ROOT, "Refusing to reset the repository root."
assert DATA_DIR != Path(DATA_DIR.anchor), "Refusing to reset a filesystem root."
is_project_data_dir = DATA_DIR == (PROJECT_ROOT / "data").resolve()
assert is_project_data_dir or ALLOW_EXTERNAL_DATA_RESET, (
    "DATA_DIR points outside PROJECT_ROOT/data. Set ALLOW_EXTERNAL_DATA_RESET=true only after reviewing the path."
)

def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

MANAGED_DIRS = [
    DATA_DIR / "bronze" / "eea",
    DATA_DIR / "bronze" / "open_meteo_raw",
    DATA_DIR / "bronze" / "wikipedia_html",
    DATA_DIR / "bronze",
    DATA_DIR / "silver",
    DATA_DIR / "gold",
    DATA_DIR / "samples",
    DATA_DIR / "checkpoints",
    DATA_DIR,
]

print(f"PROJECT_ROOT              : {PROJECT_ROOT}")
print(f"DATA_DIR                  : {DATA_DIR}")
print(f"DRY_RUN                   : {DRY_RUN}")
print(f"ALLOW_EXTERNAL_DATA_RESET : {ALLOW_EXTERNAL_DATA_RESET}")

## Implementation

### Step 1 — Preview files to be deleted

In [ ]:
import pandas as pd

all_files = sorted(DATA_DIR.rglob("*")) if DATA_DIR.exists() else []
file_records = [
    {
        "path": display_path(p),
        "size_bytes": p.stat().st_size,
        "type": p.suffix or "(no ext)",
    }
    for p in all_files
    if p.is_file()
]

preview_df = pd.DataFrame(file_records) if file_records else pd.DataFrame(columns=["path", "size_bytes", "type"])
print(f"{len(preview_df)} Datei(en) werden {'VORSCHAU (DRY_RUN)' if DRY_RUN else 'gelöscht'}:")
preview_df

### Step 2 — Delete all files, keep directory structure

In [ ]:
deleted = []
skipped = []

for p in sorted(DATA_DIR.rglob("*"), reverse=True) if DATA_DIR.exists() else []:
    if not p.is_file():
        continue
    if DRY_RUN:
        skipped.append(display_path(p))
    else:
        p.unlink()
        deleted.append(display_path(p))

if DRY_RUN:
    print(f"DRY RUN — {len(skipped)} Datei(en) würden gelöscht, keine Änderung.")
else:
    print(f"{len(deleted)} Datei(en) gelöscht.")

# Ensure all expected directories still exist
for d in MANAGED_DIRS:
    d.mkdir(parents=True, exist_ok=True)

## Validation / Quality Checks

Confirm that no files remain under `data/` and that all expected directories are present.

In [ ]:
remaining_files = [p for p in DATA_DIR.rglob("*") if p.is_file()]

if not DRY_RUN:
    assert remaining_files == [], f"Noch {len(remaining_files)} Datei(en) vorhanden: {remaining_files[:5]}"

for d in MANAGED_DIRS:
    assert d.exists() and d.is_dir(), f"Verzeichnis fehlt nach Reset: {d}"

status = "DRY RUN — keine Änderung" if DRY_RUN else f"Reset abgeschlossen. {len(deleted)} Datei(en) gelöscht."
print(status)

dir_summary = pd.DataFrame(
    [{"directory": display_path(d), "exists": d.exists()} for d in MANAGED_DIRS]
)
dir_summary

## Results

With the default `DRY_RUN=true`, the notebook lists the reset scope without changing files. With an explicitly enabled deletion run, the configured data tree becomes empty and notebooks `01` to `08` can rebuild Bronze, Silver, Gold and sample outputs.

## Limitations

- Real EEA source files placed manually under `data/bronze/eea/` are also deleted during an enabled deletion run. Keep a source copy outside the generated-data folder.
- External `DATA_DIR` paths require the additional `ALLOW_EXTERNAL_DATA_RESET=true` safeguard.